# 1. Processing the data

This notebook turns the raw model files and ERA5 into the seasonal means that the other notebooks start from. It runs on JASMIN, once per variable.

| Step | From | To (under `SCRATCH`, then `DATA_DIR`) |
|---|---|---|
| 1. LESFMIP to monthly stores | one netCDF file per member | `monthly/lesfmip/<variable>/<model>_<variable>_<experiment>_interp_monthly.zarr` |
| 2. ERA5 to monthly means | hourly ERA5 analyses | `monthly/era5_<variable>.zarr`, on the LESFMIP grid |
| 3. Seasonal means | the monthly stores | `seasonal/lesfmip_<variable>.zarr` and `seasonal/era5_<variable>.zarr` |
| 4. Test sample | the monthly stores | `tests/data/*_sample.nc` in this repository |

**Where things are written.** Dask writes zarr stores from many workers at once, which the group workspaces do not support. So everything is written under `paths.SCRATCH` and moved to `paths.DATA_DIR` by hand (the last section has the commands). Everything that reads data looks in `DATA_DIR` first and then in `SCRATCH`, so a store can be used as soon as it is written.

Steps 1 and 2 take hours and only need running again for a new variable or new models; both skip what already exists. Sections marked (JASMIN) need the data there. Section 3 can also run on the small sample in `tests/data`, which is a quick way to see what each step does.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..', '../../extant-functions']   # this repository (the extant package) and plotting_modules

import logging
import warnings
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from dask.distributed import wait
from plotting_modules import formatting, maps
from extant import config, loading, paths, preprocessing as pp, storage
from extant.logger import setup_logging
from extant.plots import methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Dask warns about every large task graph; the graphs here are large on purpose
warnings.filterwarnings('ignore', message='.*Sending large graph of size.*', category=UserWarning)

from importlib.util import find_spec

from extant import convert, datatree

#(t): flox speeds up groupby/resample on dask (the seasonal means); xarray uses it automatically when installed
if find_spec('flox') is None:
    logger.warning('flox is not installed: pip install flox to speed up the seasonal means')

## Settings

`VARIABLE` is the CMIP name of the variable to process. Its units, and how to convert them, are in `config.VARIABLES`: temperature goes from K to °C, $T_{°C} = T_K - 273.15$, and precipitation from kg m⁻² s⁻¹ to mm day⁻¹, $P_{mm\,day^{-1}} = 86400\,P_{kg\,m^{-2}\,s^{-1}}$.

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
var

In [ ]:
#(t): Where this session reads from and writes to
print(f'reading raw LESFMIP from {paths.LESFMIP_RAW / VARIABLE}')
print(f'writing to               {paths.SCRATCH}')
print(f'reading outputs from     {paths.DATA_DIR}, then {paths.SCRATCH}')

## Dask cluster (JASMIN)

The heavy steps run on a Dask Gateway cluster on JASMIN. `jasmin.stop_all(gateway)` stops every cluster left over from an earlier kernel.

In [ ]:
#(t): Start a Dask Gateway cluster, or reuse the one already running, and send it the extant package
#(c): Re-run upload_package after editing anything in extant/: the workers run its functions
from extant import jasmin

gateway, cluster, client = jasmin.connect(n_workers=20, wait_for=6)
jasmin.upload_package(client)
client

## 1. LESFMIP: member files to monthly stores (JASMIN)

The raw files are laid out as `<variable>/<experiment>/<model>/*.nc`, one file per member. Each model's members for one experiment are opened together and stacked along a new `member` dimension, labelled from the file names (`r1i1p1f1` and the like; `convert.member_from_path`). Only the area south of 40°S is kept.

First, which models have which experiments. Only models with at least `config.MIN_EXPERIMENTS` of the experiments are converted.

In [ ]:
#(t): Models with raw files for each experiment, the best-covered first
availability = convert.availability(VARIABLE, experiments=config.MAIN_EXPERIMENTS)
availability.assign(total=availability.sum(axis=1))

In [ ]:
convert_models = list(availability.index[availability.sum(axis=1) >= config.MIN_EXPERIMENTS])
convert_models

Now convert them. A store that already exists (in either place) is skipped, so this can be re-run after an interruption; `overwrite=True` redoes everything. A model whose files will not open together is reported in `failures` rather than stopping the loop.

In [ ]:
%%time
#(t): Every model and experiment to monthly stores (hours, the first time)
failures = convert.convert_lesfmip(VARIABLE, convert_models, config.MAIN_EXPERIMENTS, overwrite=False)
{key: {group: str(failure['error'])[:200] for group, failure in groups.items()} for key, groups in failures.items()}

A failed group can be retried after removing its broken files. For example, member 55 of historical GISS-E2-1-G has no time axis:

```python
failed = failures[('historical', 'GISS-E2-1-G')]['interp']
broken = failed['files'].pop(55)
convert.save_members(failed['files'], failed['store'], VARIABLE)
```

## 2. ERA5: hourly files to monthly means (JASMIN)

Each month of hourly analyses is averaged on a worker (summing in float64), then the month is regridded **conservatively** onto the LESFMIP 2.5° grid with xesmf, so each model grid cell gets the area-weighted mean of the ERA5 cells inside it. Each year is saved as soon as it is done, so an interrupted run picks up where it stopped. At the end the years are combined into one store and checked (every month once, in order).

ERA5 is kept on its own (standard) calendar. Its seasons are matched to the models' by their (year, season) labels in step 3, not by date.

In [ ]:
%%time
#(t): The LESFMIP grid to regrid onto: any monthly store will do
target = loading.open_lesfmip_monthly(VARIABLE, groups=[config.GROUP]).leaves[0].to_dataset()
era5_monthly = convert.era5_monthly(VARIABLE, target, years=range(config.ERA5_START_YEAR, 2025), client=client)
era5_monthly

## 3. Monthly to seasonal means

### Open the monthly data

Run **one** of the next two sections. Both leave the same variables: `lesfmip_tree`, monthly LESFMIP as a DataTree laid out `/<model>/<experiment>`, and `era5_monthly`, both in the units of the raw files.

#### Sample data

The 3×3 block of grid cells around `config.POINT` in `tests/data`: two models, every experiment.

In [ ]:
lesfmip_tree = loading.open_lesfmip_sample(experiments=config.MAIN_EXPERIMENTS)
era5_monthly = loading.open_era5_sample()
lesfmip_tree

#### Full data (JASMIN)

The `config.N_MODELS` models with stores for the most experiments.

In [ ]:
#(t): Which models have a monthly store for which experiment
stores = loading.store_table(VARIABLE, groups=[config.GROUP])
stores

In [ ]:
models = loading.models_with_most_experiments(stores, config.N_MODELS, experiments=config.MAIN_EXPERIMENTS)
lesfmip_tree = loading.open_lesfmip_monthly(VARIABLE, models=models, experiments=config.MAIN_EXPERIMENTS,
                                            groups=[config.GROUP])
era5_monthly = loading.open_era5_monthly(VARIABLE)
lesfmip_tree

### What is there

How many members each model ran for each experiment. A model's members differ only in their initial conditions, so the spread across them is the model's internal variability, and the more members, the better the forced response and the variability are pinned down.

In [ ]:
member_counts = pd.Series({(node.parent.name, node.name): node.sizes['member'] for node in lesfmip_tree.leaves})
member_counts = member_counts.unstack()[[e for e in config.MAIN_EXPERIMENTS if e in member_counts.index.levels[1]]]
fig, ax = methods.member_count_heatmap(member_counts, title='LESFMIP members per model and experiment')

In [ ]:
#(t): Every node's dimensions, size in memory and chunking
pd.DataFrame.from_dict({node.path: datatree.coord_info(node.to_dataset()) for node in lesfmip_tree.leaves},
                       orient='index')

In [ ]:
#(t): One month of one member, to see the domain (everything south of 40°S)
months = var.to_analysis_units(lesfmip_tree[config.PLOT_SEL['model']][config.PLOT_SEL['experiment']][VARIABLE]
                                .isel(member=0, time=[0, 6])).load()
levels = np.linspace(float(months.quantile(0.02)), float(months.quantile(0.98)), 15)
panels = maps.polar_grid(months, col_dim='time', levels=levels, cmap='RdYlBu_r',
                         label_fmt={'time': lambda t: str(t)[:7]}, cbar_label=var.label, tag=False)

### How a seasonal mean is made

Each season is the mean of its three monthly means (each month counts equally, whatever its length):

$$
\bar{x}_{\mathrm{DJF},\,y} = \tfrac{1}{3}\left(x_{\mathrm{Dec}\,y} + x_{\mathrm{Jan}\,y+1} + x_{\mathrm{Feb}\,y+1}\right),
\qquad
\bar{x}_{\mathrm{MAM},\,y} = \tfrac{1}{3}\left(x_{\mathrm{Mar}\,y} + x_{\mathrm{Apr}\,y} + x_{\mathrm{May}\,y}\right), \ \ldots
$$

so DJF is labelled by the year of its December, and a "year" runs from March to the next February. Seasons with fewer than three months are dropped (the January–February stub at the start of a record), and so is any year without all four seasons. Every experiment is first cut at the end of 2014, where historical ends, so the record ends with DJF 2013 and "the final 21 years" are the same years in every experiment. The time axis is then split into `year` × `season` (`pp.split_time`), so that a rolling window moves through the years of one season.

The figure: one member at `config.POINT`, monthly means (grey) and the seasonal means made from them.

In [ ]:
#(t): One member at the example point, monthly (in analysis units) and seasonal
point_monthly = pp.to_analysis_units(lesfmip_tree[config.PLOT_SEL['model']][config.PLOT_SEL['experiment']]
                                     .to_dataset().sel(**config.POINT), VARIABLE)[VARIABLE].isel(member=0).load()
point_seasonal = pp.seasonal_means(point_monthly.sel(time=slice(None, config.LAST_MONTH)))
fig, ax = methods.seasonal_mean_demo(point_monthly, point_seasonal, slice(1990, 1995), units=var.units)

In [ ]:
#(t): The same seasonal means, before and after splitting time into year x season
fig, axes = methods.year_season_demo(point_seasonal, units=var.units)

### All of it

`pp.seasonal_tree` runs these steps on every model and experiment, and `pp.seasonal_era5` on ERA5, which it also puts on the LESFMIP grid (`pp.match_grid`) and starts in 1979 (`config.ERA5_START_YEAR`), the start of the satellite era. Both are lazy: nothing is computed until the result is saved or loaded.

In [ ]:
lesfmip_season_tree = pp.seasonal_tree(lesfmip_tree, VARIABLE)
era5_seasonal = pp.seasonal_era5(era5_monthly, VARIABLE, like=lesfmip_season_tree)
lesfmip_season_tree

### Save (JASMIN)

Writing computes everything, in parallel on the cluster. The other notebooks then open these two stores in seconds.

In [ ]:
%%time
storage.save(lesfmip_season_tree, paths.seasonal('lesfmip', VARIABLE))
storage.save(era5_seasonal, paths.seasonal('era5', VARIABLE))

In [ ]:
#(t): Reopen what was saved, for the checks below
lesfmip_season_tree = loading.open_seasonal(VARIABLE)
era5_seasonal = loading.open_era5_seasonal(VARIABLE)

### Checks

The seasonal means at the example point: the members of one model (grey) and ERA5 (black) over the years they share. ERA5 should sit among the members; a constant offset would be a mean-state bias of the model, which the ERA5 notebook tests properly.

In [ ]:
#(t): ERA5 among one model's historical members at the example point, every season
point_tree = lesfmip_season_tree[config.PLOT_SEL['model']][config.PLOT_SEL['experiment']][VARIABLE].sel(**config.POINT)
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharex=True, layout='constrained')
for ax, season in zip(axes, ['DJF', 'MAM', 'JJA', 'SON']):
    members = point_tree.sel(season=season, year=slice(config.ERA5_START_YEAR, None))
    ax.plot(members.year, members.transpose('year', 'member'), color='0.6', lw=0.4, alpha=0.5)
    ax.plot(era5_seasonal.year, era5_seasonal.sel(**config.POINT).sel(season=season), color='k', lw=1.5)
    ax.set_title(season)
    ax.set_xlabel('Year')
axes[0].set_ylabel(var.label)
fig.suptitle(f"{config.PLOT_SEL['model']} {config.PLOT_SEL['experiment']} members (grey) and ERA5 (black) "
             f"at {formatting.format_latlon(config.POINT)}");

## 4. Test sample (JASMIN)

The 3×3 block of grid cells around `config.POINT`, from two models with every experiment and from ERA5, saved to `tests/data`. The tests and the sample sections of every notebook use it. It is temperature only, so only re-run this for `tas`.

In [ ]:
sample_tree = loading.open_lesfmip_monthly(VARIABLE, models=['CanESM5', 'HadGEM3-GC31-LL'], groups=[config.GROUP])
written = loading.write_sample(sample_tree, loading.open_era5_monthly(VARIABLE))
[f'{f.name}: {f.stat().st_size / 1e6:.1f} MB' for f in written]

## Moving the outputs to the group workspace

Once a store is written and checked, move it from scratch to the group workspace, keeping the same layout, e.g.

```bash
mkdir -p /gws/ssde/j25b/leader_epesc/aborow/ExtAnt/seasonal
mv /work/scratch-pw4/aborow/ExtAnt/seasonal/lesfmip_tas.zarr /gws/ssde/j25b/leader_epesc/aborow/ExtAnt/seasonal/
```

or everything at once with `rsync -a --remove-source-files /work/scratch-pw4/aborow/ExtAnt/ /gws/ssde/j25b/leader_epesc/aborow/ExtAnt/`. Nothing needs changing in the notebooks afterwards: they read from the group workspace first. If a store is rewritten later, the new copy lands on scratch again and a warning says the old one in the workspace is still read first until it is replaced.